# E6 - Preprocesamiento axial Al-Kafri/Sudirman: pairing imagen-ground truth

Este notebook usa los inventarios del notebook 12 para construir un primer emparejamiento trazable entre imagenes axiales `.ima` y archivos reales de ground truth. No entrena modelos y no fuerza emparejamientos ambiguos.

Objetivo: dejar candidatos, evidencia visual, sanity checks y, solo si corresponde, un `pairing_v1` procesado para que el notebook 14 pueda entrenar un primer baseline axial.

In [ ]:
# Dependencias para Google Colab.
try:
    import pydicom  # noqa: F401
    import skimage  # noqa: F401
except Exception:
    %pip install -q pydicom scikit-image

In [ ]:
import json
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
from PIL import Image
from scipy.io import loadmat, whosmat
from skimage.transform import resize
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 180)

## Montaje de Drive y rutas

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("No se monto Google Drive automaticamente. Si estas en Colab, montalo manualmente.")
    print("Detalle:", repr(exc))

In [ ]:
ALKAFRI_ROOT = Path("/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI")
EXTRACTED_ROOT = ALKAFRI_ROOT / "extracted"
NESTED_EXTRACTED_ROOT = EXTRACTED_ROOT / "_nested"
INVENTORY_RESULTS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory")
PAIRING_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E6_alkafri_pairing")
PROCESSED_ROOT = ALKAFRI_ROOT / "processed"
FIGURES_ROOT = Path("/content/drive/MyDrive/PFI_MVP/figures")
DOCS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/docs")

PAIRING_V1_ROOT = PROCESSED_ROOT / "pairing_v1"

for path in [PAIRING_ROOT, PROCESSED_ROOT, FIGURES_ROOT, DOCS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

INPUTS = {
    "extracted_inventory": INVENTORY_RESULTS_ROOT / "E6_alkafri_extracted_file_inventory.csv",
    "series_orientation": INVENTORY_RESULTS_ROOT / "E6_alkafri_series_orientation_candidates.csv",
    "ground_truth_inventory": INVENTORY_RESULTS_ROOT / "E6_alkafri_ground_truth_inventory.csv",
    "dicom_metadata_sample": INVENTORY_RESULTS_ROOT / "E6_alkafri_dicom_metadata_sample.csv",
    "dicom_reading_report": INVENTORY_RESULTS_ROOT / "E6_alkafri_dicom_reading_report.csv",
    "inventory_report": INVENTORY_RESULTS_ROOT / "E6_alkafri_inventory_report.json",
}

missing_inputs = {name: str(path) for name, path in INPUTS.items() if not path.exists()}
if missing_inputs:
    raise FileNotFoundError(missing_inputs)

print("ALKAFRI_ROOT:", ALKAFRI_ROOT)
print("PAIRING_ROOT:", PAIRING_ROOT)
print("PAIRING_V1_ROOT:", PAIRING_V1_ROOT)

## Carga de inventarios del notebook 12

In [ ]:
extracted_inventory_df = pd.read_csv(INPUTS["extracted_inventory"])
series_orientation_df = pd.read_csv(INPUTS["series_orientation"])
ground_truth_inventory_df = pd.read_csv(INPUTS["ground_truth_inventory"])
dicom_metadata_sample_df = pd.read_csv(INPUTS["dicom_metadata_sample"])
dicom_reading_report_df = pd.read_csv(INPUTS["dicom_reading_report"])
with open(INPUTS["inventory_report"], "r", encoding="utf-8") as f:
    inventory_report = json.load(f)

for df in [extracted_inventory_df, series_orientation_df, ground_truth_inventory_df]:
    if "extension" in df.columns:
        df["extension"] = df["extension"].astype(str).str.lower()

summary = {
    "total_ima_inventory": int((extracted_inventory_df.get("extension", pd.Series(dtype=str)) == ".ima").sum()),
    "valid_dicom_orientation_rows": int(series_orientation_df.get("valid_medical_dicom", pd.Series(dtype=bool)).fillna(False).astype(bool).sum()) if "valid_medical_dicom" in series_orientation_df.columns else int(len(series_orientation_df)),
    "axial_candidates": int((series_orientation_df.get("orientation_candidate", pd.Series(dtype=str)) == "axial_candidate").sum()) if "orientation_candidate" in series_orientation_df.columns else 0,
    "sagittal_candidates": int((series_orientation_df.get("orientation_candidate", pd.Series(dtype=str)) == "sagittal_candidate").sum()) if "orientation_candidate" in series_orientation_df.columns else 0,
    "ground_truth_files": int(len(ground_truth_inventory_df)),
    "png_inventory": int((extracted_inventory_df.get("extension", pd.Series(dtype=str)) == ".png").sum()),
    "mat_inventory": int((extracted_inventory_df.get("extension", pd.Series(dtype=str)) == ".mat").sum()),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

## Filtrado de imagenes axiales utiles

In [ ]:
def text_has_any(value, keywords):
    text = str(value or "").lower()
    return any(k.lower() in text for k in keywords)


def combined_text(row):
    parts = []
    for col in ["relative_path", "file_path", "SeriesDescription", "ProtocolName", "parent_folder"]:
        if col in row and pd.notna(row[col]):
            parts.append(str(row[col]))
    return " ".join(parts).lower()


if "valid_medical_dicom" not in series_orientation_df.columns:
    series_orientation_df["valid_medical_dicom"] = True
if "extension" not in series_orientation_df.columns:
    series_orientation_df["extension"] = ""

axial_ima_df = series_orientation_df[
    series_orientation_df["valid_medical_dicom"].fillna(False).astype(bool)
    & series_orientation_df["orientation_candidate"].eq("axial_candidate")
    & series_orientation_df["extension"].astype(str).str.lower().eq(".ima")
].copy()

if len(axial_ima_df) > 0:
    axial_ima_df["combined_text"] = axial_ima_df.apply(combined_text, axis=1)
    axial_ima_df["is_localizer"] = axial_ima_df["combined_text"].str.contains("localizer|localiser|survey", case=False, regex=True, na=False)
    axial_ima_df["priority_text_hit"] = axial_ima_df["combined_text"].str.contains("t2_tse_tra|t1_tse_tra|posdisp|axial|\btra\b|\bax\b", case=False, regex=True, na=False)
    axial_ima_df["sequence_candidate"] = np.select(
        [
            axial_ima_df["combined_text"].str.contains("t2", case=False, na=False),
            axial_ima_df["combined_text"].str.contains("t1", case=False, na=False),
        ],
        ["T2", "T1"],
        default="unknown",
    )
    axial_ima_df = axial_ima_df[~axial_ima_df["is_localizer"]].copy()
    axial_ima_df = axial_ima_df.sort_values(["priority_text_hit", "SeriesInstanceUID", "InstanceNumber"], ascending=[False, True, True])
else:
    axial_ima_df["combined_text"] = []
    axial_ima_df["is_localizer"] = []
    axial_ima_df["priority_text_hit"] = []
    axial_ima_df["sequence_candidate"] = []

axial_ima_candidates_csv_path = PAIRING_ROOT / "E6_alkafri_axial_ima_candidates.csv"
axial_ima_df.to_csv(axial_ima_candidates_csv_path, index=False)

print("Axial IMA candidates:", len(axial_ima_df))
print("CSV:", axial_ima_candidates_csv_path)
display(axial_ima_df.head())

## Agrupacion por paciente/serie

In [ ]:
def parse_number_list(values):
    parsed = []
    for value in values:
        try:
            if pd.notna(value):
                parsed.append(int(float(value)))
        except Exception:
            pass
    return sorted(set(parsed))


def common_path(paths):
    paths = [str(p) for p in paths if pd.notna(p)]
    if not paths:
        return ""
    return os.path.commonpath(paths)


series_rows = []
if len(axial_ima_df) > 0:
    for keys, group in axial_ima_df.groupby(["PatientID", "StudyInstanceUID", "SeriesInstanceUID", "SeriesDescription"], dropna=False):
        patient_id, study_uid, series_uid, series_desc = keys
        instance_numbers = parse_number_list(group.get("InstanceNumber", []))
        row = {
            "PatientID": patient_id,
            "StudyInstanceUID": study_uid,
            "SeriesInstanceUID": series_uid,
            "SeriesDescription": series_desc,
            "n_slices": int(len(group)),
            "Rows": group["Rows"].dropna().astype(str).mode().iloc[0] if "Rows" in group and group["Rows"].notna().any() else None,
            "Columns": group["Columns"].dropna().astype(str).mode().iloc[0] if "Columns" in group and group["Columns"].notna().any() else None,
            "PixelSpacing": group["PixelSpacing"].dropna().astype(str).mode().iloc[0] if "PixelSpacing" in group and group["PixelSpacing"].notna().any() else None,
            "SliceThickness": group["SliceThickness"].dropna().astype(str).mode().iloc[0] if "SliceThickness" in group and group["SliceThickness"].notna().any() else None,
            "InstanceNumber_min": min(instance_numbers) if instance_numbers else None,
            "InstanceNumber_max": max(instance_numbers) if instance_numbers else None,
            "InstanceNumber_unique": json.dumps(instance_numbers),
            "files_ordered": json.dumps(group.sort_values("InstanceNumber", na_position="last")["file_path"].tolist(), ensure_ascii=False),
            "relative_paths_ordered": json.dumps(group.sort_values("InstanceNumber", na_position="last")["relative_path"].tolist(), ensure_ascii=False),
            "common_path": common_path(group["file_path"].tolist()),
            "sequence_candidate": group["sequence_candidate"].dropna().astype(str).mode().iloc[0] if "sequence_candidate" in group and len(group) else "unknown",
            "possible_level": None,
        }
        level_match = re.search(r"(l[1-5]|s1)", " ".join(group["relative_path"].astype(str)).lower())
        if level_match:
            row["possible_level"] = level_match.group(1).upper()
        series_rows.append(row)

axial_series_summary_df = pd.DataFrame(series_rows)
axial_series_summary_csv_path = PAIRING_ROOT / "E6_alkafri_axial_series_summary.csv"
axial_series_summary_df.to_csv(axial_series_summary_csv_path, index=False)

print("Series axiales:", len(axial_series_summary_df))
display(axial_series_summary_df.head())

## Ground truth real desde ZIPs internos

In [ ]:
def is_real_gt_row(row):
    rel = str(row.get("relative_path", "")).lower()
    path = str(row.get("file_path", "")).lower()
    ext = str(row.get("extension", "")).lower()
    name = str(row.get("file_name", "")).lower()
    in_nested_gt = (
        "_nested" in path
        and ("ground_truth__manual_label_data" in path or "ground_truth__ground_truth_label" in path)
    )
    relevant_ext = ext in [".png", ".mat", ".xcf", ".csv", ".jpg", ".jpeg", ".bmp"]
    not_source = "source_code" not in path
    not_aux_figure = not name.startswith("figure")
    return in_nested_gt and relevant_ext and not_source and not_aux_figure


gt_real_df = extracted_inventory_df[extracted_inventory_df.apply(is_real_gt_row, axis=1)].copy()
gt_real_csv_path = PAIRING_ROOT / "E6_alkafri_ground_truth_real_files.csv"
gt_real_df.to_csv(gt_real_csv_path, index=False)

print("Ground truth real files:", len(gt_real_df))
display(gt_real_df.head())

## Exploracion de estructura de ground truth

In [ ]:
TOKEN_REGEX = re.compile(r"\d+|l[1-5]|s1", flags=re.IGNORECASE)


def numeric_tokens_from_text(text):
    return TOKEN_REGEX.findall(str(text))


def infer_patient_token(path_text):
    tokens = numeric_tokens_from_text(path_text)
    return tokens[0] if tokens else None


def infer_gt_type(path_text):
    text = str(path_text).lower()
    if "manual" in text:
        return "manual_label"
    if "ground_truth" in text or "ground truth" in text:
        return "final_ground_truth"
    if "intermedi" in text:
        return "intermediary_ground_truth"
    return "unknown"


def infer_structure_label(path_text):
    text = str(path_text).lower()
    hits = [k for k in ["disc", "posterior", "thecal", "canal", "intervertebral", "mask", "label"] if k in text]
    return "|".join(hits) if hits else "unknown"


gt_token_rows = []
for _, row in gt_real_df.iterrows():
    text = f"{row.get('relative_path', '')} {row.get('parent_folder', '')} {row.get('file_name', '')}"
    gt_token_rows.append({
        "file_path": row.get("file_path"),
        "relative_path": row.get("relative_path"),
        "file_name": row.get("file_name"),
        "extension": row.get("extension"),
        "patient_id_candidate": infer_patient_token(text),
        "numeric_tokens": json.dumps(numeric_tokens_from_text(text)),
        "gt_type": infer_gt_type(text),
        "structure_candidate": infer_structure_label(text),
        "possible_level": "|".join([t.upper() for t in numeric_tokens_from_text(text) if re.match(r"l[1-5]|s1", t, re.I)]),
    })

gt_path_tokens_df = pd.DataFrame(gt_token_rows)
gt_path_tokens_csv_path = PAIRING_ROOT / "E6_alkafri_ground_truth_path_tokens.csv"
gt_path_tokens_df.to_csv(gt_path_tokens_csv_path, index=False)

gt_folder_summary_df = (
    gt_path_tokens_df.groupby(["gt_type", "structure_candidate", "extension"], dropna=False)
    .agg(n_files=("file_path", "count"))
    .reset_index()
    if len(gt_path_tokens_df) else pd.DataFrame(columns=["gt_type", "structure_candidate", "extension", "n_files"])
)
gt_folder_summary_csv_path = PAIRING_ROOT / "E6_alkafri_ground_truth_folder_summary.csv"
gt_folder_summary_df.to_csv(gt_folder_summary_csv_path, index=False)

display(gt_path_tokens_df.head())
display(gt_folder_summary_df.head())

## Lectura de archivos MAT

In [ ]:
mat_rows = []
mat_files = gt_real_df[gt_real_df["extension"].eq(".mat")].copy() if len(gt_real_df) else pd.DataFrame()
for _, row in tqdm(mat_files.iterrows(), total=len(mat_files), desc="Leyendo MAT"):
    path = row["file_path"]
    try:
        data = loadmat(path)
        for key, value in data.items():
            if key in {"__header__", "__version__", "__globals__"}:
                continue
            arr = np.asarray(value)
            mat_rows.append({
                "file_path": path,
                "relative_path": row.get("relative_path"),
                "variable": key,
                "shape": str(tuple(arr.shape)),
                "dtype": str(arr.dtype),
                "ndim": int(arr.ndim),
                "numeric": bool(np.issubdtype(arr.dtype, np.number)),
                "mask_candidate": bool(np.issubdtype(arr.dtype, np.number) and arr.ndim in [2, 3]),
                "error": None,
            })
    except Exception as exc:
        mat_rows.append({
            "file_path": path,
            "relative_path": row.get("relative_path"),
            "variable": None,
            "shape": None,
            "dtype": None,
            "ndim": None,
            "numeric": False,
            "mask_candidate": False,
            "error": repr(exc),
        })

mat_variables_summary_df = pd.DataFrame(mat_rows)
mat_variables_summary_csv_path = PAIRING_ROOT / "E6_alkafri_mat_variables_summary.csv"
mat_variables_summary_df.to_csv(mat_variables_summary_csv_path, index=False)
display(mat_variables_summary_df.head())

## Lectura de PNG ground truth

In [ ]:
PNG_PIXEL_SAMPLE_N = 750
RANDOM_SEED = 42

png_gt_df = gt_real_df[
    gt_real_df["extension"].isin([".png", ".jpg", ".jpeg", ".bmp"])
].copy() if len(gt_real_df) else pd.DataFrame()

print("PNG/JPG/BMP ground truth detectados:", len(png_gt_df))
print("Se leeran headers de todos y pixeles solo de una muestra de:", PNG_PIXEL_SAMPLE_N)

# 1) Lectura rapida de headers
header_rows = []

for _, row in tqdm(
    png_gt_df.iterrows(),
    total=len(png_gt_df),
    desc="Leyendo headers PNG GT",
):
    item = row.to_dict()
    path = row["file_path"]

    item.update({
        "shape": None,
        "mode": None,
        "width": None,
        "height": None,
        "bands": None,
        "unique_count_approx": None,
        "unique_values_preview": None,
        "appears_binary": False,
        "appears_multiclass": False,
        "pixel_stats_computed": False,
        "pixel_stats_error": None,
        "error": None,
    })

    try:
        with Image.open(path) as img:
            width, height = img.size
            mode = img.mode
            bands = len(img.getbands())
            shape = [height, width] if bands == 1 else [height, width, bands]

            item.update({
                "shape": json.dumps(shape),
                "mode": mode,
                "width": int(width),
                "height": int(height),
                "bands": int(bands),
            })

    except Exception as exc:
        item["error"] = repr(exc)

    header_rows.append(item)

png_ground_truth_summary_df = pd.DataFrame(header_rows)

# 2) Analisis de pixeles solo en muestra configurable
valid_png_for_pixels_df = png_ground_truth_summary_df[
    png_ground_truth_summary_df["error"].isna()
].copy() if len(png_ground_truth_summary_df) else pd.DataFrame()

sample_n = min(PNG_PIXEL_SAMPLE_N, len(valid_png_for_pixels_df))
png_sample_df = (
    valid_png_for_pixels_df.sample(n=sample_n, random_state=RANDOM_SEED).copy()
    if sample_n > 0 else valid_png_for_pixels_df.copy()
)

pixel_rows = []

for _, row in tqdm(
    png_sample_df.iterrows(),
    total=len(png_sample_df),
    desc="Analizando pixeles PNG GT - muestra",
):
    path = row["file_path"]

    pixel_item = {
        "file_path": path,
        "unique_count_approx": None,
        "unique_values_preview": None,
        "appears_binary": False,
        "appears_multiclass": False,
        "pixel_stats_computed": True,
        "pixel_stats_error": None,
    }

    try:
        with Image.open(path) as img:
            arr = np.asarray(img)

        if arr.ndim == 2:
            unique_values = np.unique(arr)
            preview = unique_values[:30].tolist()
            unique_count = int(len(unique_values))
            values_set = set(int(v) for v in unique_values[:100])
            appears_binary = unique_count <= 3 and values_set.issubset({0, 1, 255})
            appears_multiclass = unique_count > 3 and unique_count <= 30

            pixel_item.update({
                "unique_count_approx": unique_count,
                "unique_values_preview": json.dumps(preview),
                "appears_binary": bool(appears_binary),
                "appears_multiclass": bool(appears_multiclass),
            })

        else:
            flat = arr.reshape(-1, arr.shape[-1])

            max_pixels = 200_000
            if flat.shape[0] > max_pixels:
                rng = np.random.default_rng(RANDOM_SEED)
                idx = rng.choice(flat.shape[0], size=max_pixels, replace=False)
                flat = flat[idx]

            unique_colors = np.unique(flat, axis=0)
            preview = unique_colors[:20].tolist()
            unique_count = int(len(unique_colors))
            appears_binary = unique_count <= 3
            appears_multiclass = unique_count > 3 and unique_count <= 50

            pixel_item.update({
                "unique_count_approx": unique_count,
                "unique_values_preview": json.dumps(preview),
                "appears_binary": bool(appears_binary),
                "appears_multiclass": bool(appears_multiclass),
            })

    except Exception as exc:
        pixel_item["pixel_stats_error"] = repr(exc)

    pixel_rows.append(pixel_item)

pixel_stats_df = pd.DataFrame(pixel_rows)

if len(pixel_stats_df) > 0:
    cols_to_update = [
        "unique_count_approx",
        "unique_values_preview",
        "appears_binary",
        "appears_multiclass",
        "pixel_stats_computed",
        "pixel_stats_error",
    ]

    png_ground_truth_summary_df = png_ground_truth_summary_df.merge(
        pixel_stats_df[["file_path"] + cols_to_update],
        on="file_path",
        how="left",
        suffixes=("", "_sample"),
    )

    for col in cols_to_update:
        sample_col = f"{col}_sample"
        if sample_col in png_ground_truth_summary_df.columns:
            png_ground_truth_summary_df[col] = png_ground_truth_summary_df[sample_col].combine_first(
                png_ground_truth_summary_df[col]
            )
            png_ground_truth_summary_df.drop(columns=[sample_col], inplace=True)

if "pixel_stats_computed" in png_ground_truth_summary_df.columns:
    png_ground_truth_summary_df["pixel_stats_computed"] = (
        png_ground_truth_summary_df["pixel_stats_computed"].fillna(False).astype(bool)
    )

png_ground_truth_summary_csv_path = PAIRING_ROOT / "E6_alkafri_png_ground_truth_summary.csv"
png_ground_truth_summary_df.to_csv(png_ground_truth_summary_csv_path, index=False)

print("PNG ground truth summary:", png_ground_truth_summary_csv_path)
print("Total PNG/JPG/BMP resumidos:", len(png_ground_truth_summary_df))
print(
    "Con analisis de pixeles:",
    int(png_ground_truth_summary_df["pixel_stats_computed"].sum()) if "pixel_stats_computed" in png_ground_truth_summary_df.columns else 0,
)

display(png_ground_truth_summary_df.head())

## Tokens de imagenes y ground truth

In [ ]:
def extract_path_tokens(row):
    text = f"{row.get('relative_path', '')} {row.get('file_path', '')} {row.get('file_name', '')}"
    tokens = numeric_tokens_from_text(text)
    patient = row.get("PatientID") if "PatientID" in row and pd.notna(row.get("PatientID")) else (tokens[0] if tokens else None)
    instance = row.get("InstanceNumber") if "InstanceNumber" in row and pd.notna(row.get("InstanceNumber")) else (tokens[-1] if tokens else None)
    return {
        "file_path": row.get("file_path"),
        "relative_path": row.get("relative_path"),
        "patient_id_candidate": str(patient) if patient is not None else None,
        "series_id_candidate": row.get("SeriesInstanceUID") if "SeriesInstanceUID" in row else None,
        "slice_id_candidate": str(instance) if instance is not None else None,
        "numeric_tokens": json.dumps(tokens),
        "level_tokens": json.dumps([t.upper() for t in tokens if re.match(r"l[1-5]|s1", t, re.I)]),
    }


image_path_tokens_df = pd.DataFrame([extract_path_tokens(row) for _, row in axial_ima_df.iterrows()])
gt_path_tokens_pairing_df = pd.DataFrame([extract_path_tokens(row) for _, row in gt_real_df.iterrows()])

image_path_tokens_csv_path = PAIRING_ROOT / "E6_alkafri_image_path_tokens.csv"
gt_path_tokens_pairing_csv_path = PAIRING_ROOT / "E6_alkafri_gt_path_tokens.csv"
image_path_tokens_df.to_csv(image_path_tokens_csv_path, index=False)
gt_path_tokens_pairing_df.to_csv(gt_path_tokens_pairing_csv_path, index=False)
display(image_path_tokens_df.head())
display(gt_path_tokens_pairing_df.head())

## Estrategias de emparejamiento candidatas

In [ ]:
def parse_shape(value):
    if pd.isna(value):
        return None
    nums = [int(x) for x in re.findall(r"\d+", str(value))]
    return tuple(nums) if nums else None


def image_shape_from_row(row):
    try:
        return (int(float(row.get("Rows"))), int(float(row.get("Columns"))))
    except Exception:
        return None


gt_shape_lookup = {}
if len(png_ground_truth_summary_df):
    for _, row in png_ground_truth_summary_df.iterrows():
        shape = parse_shape(row.get("shape"))
        if shape:
            gt_shape_lookup[row["file_path"]] = shape[:2]
if len(mat_variables_summary_df):
    for _, row in mat_variables_summary_df[mat_variables_summary_df["mask_candidate"].fillna(False)].iterrows():
        shape = parse_shape(row.get("shape"))
        if shape:
            gt_shape_lookup[row["file_path"]] = shape[:2]


strategy_a_rows = []
for _, img in image_path_tokens_df.iterrows():
    img_patient = str(img.get("patient_id_candidate"))
    if not img_patient or img_patient == "nan":
        continue
    matches = gt_path_tokens_pairing_df[gt_path_tokens_pairing_df["patient_id_candidate"].astype(str).eq(img_patient)]
    for _, gt in matches.iterrows():
        strategy_a_rows.append({
            "image_file_path": img["file_path"],
            "gt_file_path": gt["file_path"],
            "patient_id_candidate": img_patient,
            "reason": "patient_or_folder_token_match",
        })
strategy_a_df = pd.DataFrame(strategy_a_rows)
strategy_a_csv_path = PAIRING_ROOT / "E6_alkafri_pairing_strategy_a_patient.csv"
strategy_a_df.to_csv(strategy_a_csv_path, index=False)

strategy_b_rows = []
for _, img in image_path_tokens_df.iterrows():
    img_tokens = set(json.loads(img.get("numeric_tokens", "[]")))
    if not img_tokens:
        continue
    for _, gt in gt_path_tokens_pairing_df.iterrows():
        gt_tokens = set(json.loads(gt.get("numeric_tokens", "[]")))
        shared = sorted(img_tokens & gt_tokens)
        if shared:
            strategy_b_rows.append({
                "image_file_path": img["file_path"],
                "gt_file_path": gt["file_path"],
                "shared_tokens": json.dumps(shared),
                "reason": "numeric_token_overlap",
            })
strategy_b_df = pd.DataFrame(strategy_b_rows)
strategy_b_csv_path = PAIRING_ROOT / "E6_alkafri_pairing_strategy_b_tokens.csv"
strategy_b_df.to_csv(strategy_b_csv_path, index=False)

strategy_c_rows = []
for _, img in axial_ima_df.iterrows():
    img_shape = image_shape_from_row(img)
    if img_shape is None:
        continue
    for gt_path, gt_shape in gt_shape_lookup.items():
        shape_match = tuple(img_shape) == tuple(gt_shape)
        if shape_match:
            strategy_c_rows.append({
                "image_file_path": img["file_path"],
                "gt_file_path": gt_path,
                "image_shape": str(img_shape),
                "mask_shape": str(gt_shape),
                "reason": "shape_match",
            })
strategy_c_df = pd.DataFrame(strategy_c_rows)
strategy_c_csv_path = PAIRING_ROOT / "E6_alkafri_pairing_strategy_c_shape.csv"
strategy_c_df.to_csv(strategy_c_csv_path, index=False)

metadata_files = extracted_inventory_df[
    extracted_inventory_df["file_name"].astype(str).str.lower().isin([
        "slices numbers.csv", "t1_subfolders.csv", "t2_subfolders.csv", "png counts.csv"
    ])
].copy()
strategy_d_rows = []
for _, row in metadata_files.iterrows():
    try:
        preview = pd.read_csv(row["file_path"], nrows=10).to_json(orient="records", force_ascii=False)
    except Exception as exc:
        preview = repr(exc)
    strategy_d_rows.append({
        "metadata_file_path": row["file_path"],
        "relative_path": row["relative_path"],
        "file_name": row["file_name"],
        "preview": preview,
        "reason": "source_metadata_hint",
    })
strategy_d_df = pd.DataFrame(strategy_d_rows)
strategy_d_csv_path = PAIRING_ROOT / "E6_alkafri_pairing_strategy_d_source_metadata.csv"
strategy_d_df.to_csv(strategy_d_csv_path, index=False)

print(len(strategy_a_df), len(strategy_b_df), len(strategy_c_df), len(strategy_d_df))

## Seleccion de pairing preliminar

In [ ]:
signals = {}

def add_signal(image_path, gt_path, signal_name):
    key = (image_path, gt_path)
    signals.setdefault(key, set()).add(signal_name)

for _, row in strategy_a_df.iterrows():
    add_signal(row["image_file_path"], row["gt_file_path"], "patient")
for _, row in strategy_b_df.iterrows():
    add_signal(row["image_file_path"], row["gt_file_path"], "tokens")
for _, row in strategy_c_df.iterrows():
    add_signal(row["image_file_path"], row["gt_file_path"], "shape")

img_lookup = axial_ima_df.set_index("file_path").to_dict(orient="index") if len(axial_ima_df) else {}
gt_lookup = gt_real_df.set_index("file_path").to_dict(orient="index") if len(gt_real_df) else {}

pair_rows = []
for (img_path, gt_path), sigs in signals.items():
    img = img_lookup.get(img_path, {})
    gt = gt_lookup.get(gt_path, {})
    image_shape = image_shape_from_row(img)
    mask_shape = gt_shape_lookup.get(gt_path)
    shape_match = bool(image_shape is not None and mask_shape is not None and tuple(image_shape) == tuple(mask_shape))
    n_signals = len(sigs)
    if {"patient", "shape", "tokens"}.issubset(sigs):
        confidence = "alta"
    elif n_signals >= 2:
        confidence = "media"
    else:
        confidence = "baja"
    needs_manual_review = confidence != "alta"
    pair_rows.append({
        "image_file_path": img_path,
        "image_relative_path": img.get("relative_path"),
        "gt_file_path": gt_path,
        "gt_relative_path": gt.get("relative_path"),
        "pairing_strategy": "|".join(sorted(sigs)),
        "patient_id_candidate": img.get("PatientID"),
        "series_id_candidate": img.get("SeriesInstanceUID"),
        "slice_id_candidate": img.get("InstanceNumber"),
        "image_shape": str(image_shape),
        "mask_shape": str(mask_shape),
        "shape_match": shape_match,
        "confidence": confidence,
        "reason": f"signals={sorted(sigs)}",
        "needs_manual_review": needs_manual_review,
    })

pairing_candidates_df = pd.DataFrame(pair_rows)
if len(pairing_candidates_df) > 0:
    pairing_candidates_df = pairing_candidates_df.sort_values(["confidence", "shape_match"], ascending=[True, False])

pairing_candidates_csv_path = PAIRING_ROOT / "E6_alkafri_pairing_candidates.csv"
pairing_candidates_df.to_csv(pairing_candidates_csv_path, index=False)

print("Pairing candidates:", len(pairing_candidates_df))
display(pairing_candidates_df.head(20))

## Visualizacion de pairing candidatos

In [ ]:
def normalize_percentile(arr, p_low=1, p_high=99):
    arr = arr.astype(np.float32)
    low, high = np.percentile(arr, [p_low, p_high])
    if np.isclose(low, high):
        return np.zeros_like(arr, dtype=np.float32)
    return np.clip((arr - low) / (high - low), 0, 1).astype(np.float32)


def read_image_ima(path):
    ds = pydicom.dcmread(str(path), force=True)
    return normalize_percentile(ds.pixel_array), ds


def read_mask_any(path):
    path = str(path)
    ext = Path(path).suffix.lower()
    if ext in [".png", ".jpg", ".jpeg", ".bmp"]:
        return np.asarray(Image.open(path))
    if ext == ".mat":
        data = loadmat(path)
        for key, value in data.items():
            if key.startswith("__"):
                continue
            arr = np.asarray(value)
            if np.issubdtype(arr.dtype, np.number) and arr.ndim in [2, 3]:
                return arr[:, :, 0] if arr.ndim == 3 else arr
    raise ValueError(f"No se pudo leer mascara: {path}")


def plot_pair(image, mask, title, output_path):
    mask_2d = mask[:, :, 0] if mask.ndim == 3 else mask
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(image, cmap="gray")
    axes[0].set_title("Imagen axial")
    axes[1].imshow(mask_2d, cmap="viridis")
    axes[1].set_title("Ground truth")
    axes[2].imshow(image, cmap="gray")
    overlay = np.ma.masked_where(mask_2d == 0, mask_2d)
    axes[2].imshow(overlay, cmap="autumn", alpha=0.45)
    axes[2].set_title("Overlay")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close(fig)


pairing_figure_paths = []
visualizable_pairs = pairing_candidates_df[pairing_candidates_df["confidence"].isin(["alta", "media"])].copy() if len(pairing_candidates_df) else pd.DataFrame()
for idx, row in enumerate(visualizable_pairs.head(10).itertuples(index=False), start=1):
    output_path = FIGURES_ROOT / f"E6_alkafri_pairing_example_{idx:02d}.png"
    try:
        image, ds = read_image_ima(row.image_file_path)
        mask = read_mask_any(row.gt_file_path)
        plot_pair(image, mask, f"{row.confidence} - {row.pairing_strategy}", output_path)
        pairing_figure_paths.append(str(output_path))
    except Exception as exc:
        print("No se pudo visualizar par:", row.image_file_path, row.gt_file_path, repr(exc))

if not pairing_figure_paths:
    axial_examples_path = FIGURES_ROOT / "E6_alkafri_unpaired_axial_image_examples.png"
    gt_examples_path = FIGURES_ROOT / "E6_alkafri_unpaired_ground_truth_examples.png"
    fig, axes = plt.subplots(1, min(5, len(axial_ima_df)), figsize=(15, 3)) if len(axial_ima_df) else (None, [])
    if len(axial_ima_df):
        axes = np.atleast_1d(axes)
        for ax, (_, row) in zip(axes, axial_ima_df.head(5).iterrows()):
            try:
                image, _ = read_image_ima(row["file_path"])
                ax.imshow(image, cmap="gray")
            except Exception:
                ax.text(0.5, 0.5, "error", ha="center")
            ax.axis("off")
        fig.tight_layout()
        fig.savefig(axial_examples_path, dpi=160, bbox_inches="tight")
        plt.close(fig)
        pairing_figure_paths.append(str(axial_examples_path))
    fig, axes = plt.subplots(1, min(5, len(gt_real_df)), figsize=(15, 3)) if len(gt_real_df) else (None, [])
    if len(gt_real_df):
        axes = np.atleast_1d(axes)
        for ax, (_, row) in zip(axes, gt_real_df.head(5).iterrows()):
            try:
                mask = read_mask_any(row["file_path"])
                ax.imshow(mask[:, :, 0] if mask.ndim == 3 else mask, cmap="viridis")
            except Exception:
                ax.text(0.5, 0.5, "error", ha="center")
            ax.axis("off")
        fig.tight_layout()
        fig.savefig(gt_examples_path, dpi=160, bbox_inches="tight")
        plt.close(fig)
        pairing_figure_paths.append(str(gt_examples_path))

print("Figuras:", pairing_figure_paths)

## Dataset axial procesado preliminar

In [ ]:
processed_rows = []
trusted_pairs = pairing_candidates_df[pairing_candidates_df["confidence"].isin(["alta", "media"])].copy() if len(pairing_candidates_df) else pd.DataFrame()
pairing_v1_created = False

if len(trusted_pairs) > 0:
    PAIRING_V1_ROOT.mkdir(parents=True, exist_ok=True)
    for idx, row in enumerate(trusted_pairs.itertuples(index=False), start=1):
        sample_id = f"pair_{idx:04d}"
        try:
            image, ds = read_image_ima(row.image_file_path)
            mask = read_mask_any(row.gt_file_path)
            image_out = PAIRING_V1_ROOT / f"{sample_id}_image.npy"
            mask_out = PAIRING_V1_ROOT / f"{sample_id}_mask.npy"
            np.save(image_out, image.astype(np.float32))
            np.save(mask_out, mask)
            processed_rows.append({
                "sample_id": sample_id,
                "image_npy": str(image_out),
                "mask_npy": str(mask_out),
                "image_file_path": row.image_file_path,
                "gt_file_path": row.gt_file_path,
                "confidence": row.confidence,
                "pairing_strategy": row.pairing_strategy,
                "image_shape": str(image.shape),
                "mask_shape": str(mask.shape),
            })
        except Exception as exc:
            processed_rows.append({
                "sample_id": sample_id,
                "image_file_path": row.image_file_path,
                "gt_file_path": row.gt_file_path,
                "confidence": row.confidence,
                "pairing_strategy": row.pairing_strategy,
                "error": repr(exc),
            })
    pairing_v1_created = any("image_npy" in row for row in processed_rows)
else:
    print("No hay pares alta/media; no se crea dataset final pairing_v1.")

processed_pairing_index_df = pd.DataFrame(processed_rows)
processed_pairing_index_csv_path = PAIRING_ROOT / "E6_alkafri_processed_pairing_v1_index.csv"
processed_pairing_index_df.to_csv(processed_pairing_index_csv_path, index=False)
display(processed_pairing_index_df.head())

## Sanity checks, reporte y conclusion

In [ ]:
confidence_counts = pairing_candidates_df["confidence"].value_counts().to_dict() if len(pairing_candidates_df) else {}
problems = []
if len(pairing_candidates_df) == 0:
    problems.append("No se encontraron pares candidatos.")
if len(pairing_candidates_df) and not pairing_candidates_df["shape_match"].any():
    problems.append("No hay pares con shape match.")
if len(strategy_d_df) > 0:
    problems.append("Se requiere interpretar metadata/codigo MATLAB para confirmar pairing.")
if len(gt_real_df) > 0 and len(gt_shape_lookup) == 0:
    problems.append("Ground truth en formato inesperado o sin shape legible.")

sanity_checks = {
    "n_axial_series": int(len(axial_series_summary_df)),
    "n_axial_ima_candidates": int(len(axial_ima_df)),
    "n_ground_truth_real": int(len(gt_real_df)),
    "n_png_ground_truth": int(len(png_ground_truth_summary_df)),
    "n_mat_ground_truth": int(len(mat_files)),
    "n_pairing_candidates": int(len(pairing_candidates_df)),
    "n_pairs_alta": int(confidence_counts.get("alta", 0)),
    "n_pairs_media": int(confidence_counts.get("media", 0)),
    "n_pairs_baja": int(confidence_counts.get("baja", 0)),
    "n_shape_match": int(pairing_candidates_df["shape_match"].sum()) if len(pairing_candidates_df) else 0,
    "n_pairs_visualized": int(len(pairing_figure_paths)),
    "pairing_v1_created": bool(pairing_v1_created),
    "problems": problems,
}
sanity_checks_json_path = PAIRING_ROOT / "E6_alkafri_pairing_sanity_checks.json"
with open(sanity_checks_json_path, "w", encoding="utf-8") as f:
    json.dump(sanity_checks, f, indent=2, ensure_ascii=False)

exports = {
    "axial_ima_candidates": str(axial_ima_candidates_csv_path),
    "axial_series_summary": str(axial_series_summary_csv_path),
    "ground_truth_real_files": str(gt_real_csv_path),
    "ground_truth_path_tokens": str(gt_path_tokens_csv_path),
    "ground_truth_folder_summary": str(gt_folder_summary_csv_path),
    "mat_variables_summary": str(mat_variables_summary_csv_path),
    "png_ground_truth_summary": str(png_ground_truth_summary_csv_path),
    "image_path_tokens": str(image_path_tokens_csv_path),
    "gt_path_tokens": str(gt_path_tokens_pairing_csv_path),
    "strategy_a": str(strategy_a_csv_path),
    "strategy_b": str(strategy_b_csv_path),
    "strategy_c": str(strategy_c_csv_path),
    "strategy_d": str(strategy_d_csv_path),
    "pairing_candidates": str(pairing_candidates_csv_path),
    "processed_pairing_v1_index": str(processed_pairing_index_csv_path),
    "sanity_checks": str(sanity_checks_json_path),
    "figures": pairing_figure_paths,
}

recommendation_14 = (
    "Entrenar notebook 14 solo con pairing_v1 alta/media y revision manual previa."
    if pairing_v1_created else
    "No entrenar todavia; resolver ambiguedad de pairing interpretando metadata y codigo MATLAB."
)

report = {
    "inputs": {k: str(v) for k, v in INPUTS.items()},
    "n_valid_dicom_ima": int(summary["valid_dicom_orientation_rows"]),
    "n_axial_candidates": int(len(axial_ima_df)),
    "n_axial_series": int(len(axial_series_summary_df)),
    "n_ground_truth": int(len(gt_real_df)),
    "strategy_results": {
        "patient": int(len(strategy_a_df)),
        "tokens": int(len(strategy_b_df)),
        "shape": int(len(strategy_c_df)),
        "source_metadata": int(len(strategy_d_df)),
    },
    "confidence_counts": confidence_counts,
    "processed_dataset_created": bool(pairing_v1_created),
    "exports": exports,
    "recommendation_for_notebook_14": recommendation_14,
}
report_json_path = PAIRING_ROOT / "E6_alkafri_pairing_report.json"
with open(report_json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("Sanity:", sanity_checks_json_path)
print("Report:", report_json_path)

In [ ]:
series_table_md = axial_series_summary_df.head(20).to_markdown(index=False) if len(axial_series_summary_df) else "Sin series axiales."
gt_table_md = gt_folder_summary_df.to_markdown(index=False) if len(gt_folder_summary_df) else "Sin ground truth real."
pairing_table_md = pairing_candidates_df.head(20).to_markdown(index=False) if len(pairing_candidates_df) else "Sin pares candidatos confiables."

conclusion_text = f'''# Conclusion tecnica - E6 Pairing axial Al-Kafri/Sudirman

## Objetivo

Construir el primer preprocesamiento axial trazable de Al-Kafri/Sudirman a partir de los inventarios del notebook 12, resolviendo candidatos de emparejamiento entre imagenes `.ima` axiales y ground truth real.

## Por que se hace despues del inventario

El notebook 12 confirmo que hay DICOM/IMA axiales validos y ground truth extraido desde ZIPs internos, pero el emparejamiento imagen-mascara no estaba resuelto. Este notebook separa exploracion de inventario y decision de pairing para evitar asumir relaciones sin evidencia.

## Metodologia de pairing

Se evaluaron estrategias por ID/carpeta de paciente, tokens numericos de path, match de dimensiones y metadata derivada de archivos auxiliares del codigo fuente. La confianza alta/media/baja se asigno segun cantidad y calidad de senales coincidentes.

## Hallazgos de imagenes axiales

{series_table_md}

## Hallazgos de ground truth

{gt_table_md}

## Resultado de pairing

{pairing_table_md}

Dataset `pairing_v1` creado: {pairing_v1_created}.

## Ejemplos visuales

{pairing_figure_paths}

## Limitaciones

- No se entrenan modelos.
- No se fuerza pairing definitivo si hay ambiguedad.
- Algunas mascaras pueden requerir interpretar `.mat` o scripts MATLAB.
- Shape match no es suficiente por si solo.
- Se requiere revision manual/profesional antes de usar pares para entrenamiento.

## Recomendacion

{recommendation_14}
'''

conclusion_path = DOCS_ROOT / "E6_alkafri_pairing_conclusion.md"
with open(conclusion_path, "w", encoding="utf-8") as f:
    f.write(conclusion_text)

print("Conclusion:", conclusion_path)

## Criterio de aceptacion

Este notebook cumple si carga los inventarios del notebook 12, filtra `.ima` axiales reales, excluye localizers/source_code, inspecciona PNG/MAT de ground truth real, propone estrategias trazables de pairing, exporta candidatos con confianza, genera visualizaciones y no entrena modelos.